In [1]:
import pandas as pd
import numpy as np
import random
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MERGED_CSV   = "webnlg_qa_merged_validated.csv"
COVERAGE_CSV = "entity_coverage_report.csv"   # entity-level (category+eid)
OUTPUT_CSV   = "webnlg_qa_selected.csv"

RANDOM_SEED  = 42
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # fast, good enough for diversity

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

CSV_FIELDNAMES = [
    "split", "category", "eid",
    "lex_id", "lex_text",
    "xml_file", "mtriple", "otriple",
    "statement",
    "question_idx", "question_type", "question", "answer",
    "generation_errors", "corrected_answer", "correction_flag",
    # selection metadata
    "selected_role",      # yes_q | no_q | extractive_1 | extractive_2
    "selection_source",   # original | corrected | random
]

# ─────────────────────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────────────────────
print("Loading data…")
df  = pd.read_csv(MERGED_CSV, dtype=str).fillna("")
cov = pd.read_csv(COVERAGE_CSV, dtype=str).fillna("")

# Only keep entities that are complete
complete_entities = cov[cov["is_complete"].str.lower() == "true"][["category", "eid"]]
print(f"  Complete entities      : {len(complete_entities)}")

df = df.merge(complete_entities, on=["category", "eid"], how="inner")
print(f"  Rows after entity filter: {len(df)}")

# ─────────────────────────────────────────────────────────────
# EFFECTIVE ANSWER — use corrected_answer if available
# ─────────────────────────────────────────────────────────────
df["effective_answer"] = df.apply(
    lambda r: r["corrected_answer"].strip()
              if r.get("corrected_answer", "").strip() != ""
              else r["answer"].strip(),
    axis=1
)

# Was this row modified by the model evaluators?
# A row is "original" if correction_flag is empty
df["is_original"] = df.get("correction_flag", pd.Series([""] * len(df))).str.strip() == ""

# ─────────────────────────────────────────────────────────────
# LOAD EMBEDDING MODEL
# ─────────────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)


def embed_questions(questions: list[str]) -> np.ndarray:
    return embedder.encode(questions, normalize_embeddings=True, show_progress_bar=False)


# ─────────────────────────────────────────────────────────────
# SYNTACTIC DISTANCE — char n-gram / edit-ratio based
# ─────────────────────────────────────────────────────────────
def syntactic_distance(a: str, b: str) -> float:
    """1 - SequenceMatcher ratio → higher = more different."""
    return 1.0 - SequenceMatcher(None, a.lower(), b.lower()).ratio()


def combined_distance(q1: str, q2: str, emb1: np.ndarray, emb2: np.ndarray,
                      sem_weight: float = 0.7) -> float:
    """Weighted combo of semantic (cosine) and syntactic distance."""
    sem  = float(1.0 - np.dot(emb1, emb2))        # cosine distance (embeddings normalised)
    syn  = syntactic_distance(q1, q2)
    return sem_weight * sem + (1 - sem_weight) * syn


def select_most_diverse_pair(rows: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """
    From a set of extractive question rows, return the pair that
    maximises combined semantic + syntactic distance.
    """
    qs = rows["question"].tolist()
    if len(qs) == 1:
        return rows.iloc[0], rows.iloc[0]   # only one available — duplicate
    if len(qs) == 2:
        return rows.iloc[0], rows.iloc[1]

    embs = embed_questions(qs)

    best_dist = -1.0
    best_i, best_j = 0, 1
    for i in range(len(qs)):
        for j in range(i + 1, len(qs)):
            d = combined_distance(qs[i], qs[j], embs[i], embs[j])
            if d > best_dist:
                best_dist = d
                best_i, best_j = i, j

    return rows.iloc[best_i], rows.iloc[best_j]


# ─────────────────────────────────────────────────────────────
# SELECTION LOGIC — priority: original → random
# ─────────────────────────────────────────────────────────────
def select_one(pool: pd.DataFrame, label: str) -> tuple[pd.Series, str]:
    """
    Select one row from pool.
    Priority: original (correction_flag == "") → random.
    Returns (row, source_label).
    """
    originals = pool[pool["is_original"]]
    if len(originals) > 0:
        return originals.sample(1, random_state=RANDOM_SEED).iloc[0], "original"
    return pool.sample(1, random_state=RANDOM_SEED).iloc[0], "random"


def select_extractive_pair(pool: pd.DataFrame) -> list[tuple[pd.Series, str]]:
    """
    Select 2 most-diverse extractive questions.
    Prioritise original rows; if < 2 originals exist, fill from corrected.
    Returns list of (row, source_label).
    """
    originals = pool[pool["is_original"]]
    corrected = pool[~pool["is_original"]]

    if len(originals) >= 2:
        r1, r2 = select_most_diverse_pair(originals)
        return [(r1, "original"), (r2, "original")]

    elif len(originals) == 1:
        # 1 original + 1 from corrected (most diverse from the original)
        orig_row  = originals.iloc[0]
        orig_emb  = embed_questions([orig_row["question"]])[0]

        if len(corrected) == 0:
            return [(orig_row, "original"), (orig_row, "original")]  # only 1 exists

        cand_qs   = corrected["question"].tolist()
        cand_embs = embed_questions(cand_qs)
        dists = [combined_distance(orig_row["question"], cand_qs[i], orig_emb, cand_embs[i])
                 for i in range(len(cand_qs))]
        best_idx = int(np.argmax(dists))
        return [(orig_row, "original"), (corrected.iloc[best_idx], "corrected")]

    else:
        # No originals — pick most diverse pair from all
        if len(pool) == 0:
            return []
        r1, r2 = select_most_diverse_pair(pool)
        return [(r1, "random"), (r2, "random")]


# ─────────────────────────────────────────────────────────────
# MAIN SELECTION LOOP
# ─────────────────────────────────────────────────────────────
print("Running selection per entity (category, eid)…")

selected_records = []

entity_groups = df.groupby(["category", "eid"])
n_entities = len(entity_groups)
n_complete = 0
n_skipped  = 0

for (cat, eid), group in entity_groups:

    # ── Split by type + answer polarity ─────────────────────
    yes_pool  = group[
        (group["question_type"] == "yes_no") &
        (group["effective_answer"].str.lower() == "yes")
    ]
    no_pool   = group[
        (group["question_type"] == "yes_no") &
        (group["effective_answer"].str.lower() == "no")
    ]
    extr_pool = group[group["question_type"] == "extractive"]

    if len(yes_pool) == 0 or len(no_pool) == 0 or len(extr_pool) < 2:
        n_skipped += 1
        continue

    # ── Select yes / no ─────────────────────────────────────
    yes_row,  yes_src  = select_one(yes_pool,  "yes_q")
    no_row,   no_src   = select_one(no_pool,   "no_q")

    # ── Select 2 diverse extractive ─────────────────────────
    extr_pair = select_extractive_pair(extr_pool)
    if len(extr_pair) < 2:
        n_skipped += 1
        continue

    # ── Build output rows ────────────────────────────────────
    for row, src, role in [
        (yes_row,        yes_src,         "yes_q"),
        (no_row,         no_src,          "no_q"),
        (extr_pair[0][0], extr_pair[0][1], "extractive_1"),
        (extr_pair[1][0], extr_pair[1][1], "extractive_2"),
    ]:
        rec = row.to_dict()
        rec["selected_role"]    = role
        rec["selection_source"] = src
        selected_records.append(rec)

    n_complete += 1


# ─────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────
out_df = pd.DataFrame(selected_records)

# Ensure all expected columns exist
for col in CSV_FIELDNAMES:
    if col not in out_df.columns:
        out_df[col] = ""

out_df = out_df[CSV_FIELDNAMES]
out_df.to_csv(OUTPUT_CSV, index=False)

# ─────────────────────────────────────────────────────────────
# REPORT
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SELECTION REPORT")
print("=" * 60)
print(f"  Total entities            : {n_entities}")
print(f"  ✅ Selected (4 Qs each)   : {n_complete}  → {n_complete * 4} rows")
print(f"  ⚠  Skipped (insufficient) : {n_skipped}")
print(f"\nSelection source breakdown:")
print(out_df["selection_source"].value_counts().to_string())
print(f"\nSelected role breakdown:")
print(out_df["selected_role"].value_counts().to_string())
print(f"\nOutput saved to: {OUTPUT_CSV}")

Loading data…
  Complete entities      : 3253
  Rows after entity filter: 60645
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Running selection per entity (category, eid)…

SELECTION REPORT
  Total entities            : 3253
  ✅ Selected (4 Qs each)   : 3253  → 13012 rows
  ⚠  Skipped (insufficient) : 0

Selection source breakdown:
selection_source
original     12479
random         517
corrected       16

Selected role breakdown:
selected_role
yes_q           3253
no_q            3253
extractive_1    3253
extractive_2    3253

Output saved to: webnlg_qa_selected.csv
